In [3]:
from IPython.display import HTML
HTML("""
<style>
.jp-OutputArea-output, .output_area pre {
    max-height: none !important;
}
.jp-OutputArea-child, .output_scroll {
    height: auto !important;
    max-height: none !important;
    overflow-y: visible !important;
    overflow-x: auto !important;
}
</style>
""")

In [2]:
# ============================================================
# GEO SAFETY MODEL SEARCH - LIGHT VERSION
# RF + sklearn MLP only
# Smaller search, cleaner outputs, CSV tracking
# ============================================================

import json
import math
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import ParameterGrid, KFold, train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
RANDOM_STATE = 42
TEST_SIZE = 0.20
N_SPLITS_CV = 5

# >>> CHANGE THIS IF NEEDED <<<
DATA_PATH = Path("data/compiled_model_ready/MR_cities_worldpop_knn_macro_v5.csv")

ARTIFACT_DIR = Path("artifacts/geo_safety_rf_mlp_light")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COL = "safety_index"

print("=" * 100)
print("GEO SAFETY RF + MLP LIGHT SEARCH")
print("=" * 100)
print(f"[CONFIG] DATA_PATH      : {DATA_PATH}")
print(f"[CONFIG] TARGET_COL     : {TARGET_COL}")
print(f"[CONFIG] TEST_SIZE      : {TEST_SIZE}")
print(f"[CONFIG] CV_FOLDS       : {N_SPLITS_CV}")
print(f"[CONFIG] OUTPUT_DIR     : {ARTIFACT_DIR}")
print("=" * 100)

# ------------------------------------------------------------
# LOAD DATA
# ------------------------------------------------------------
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

df = pd.read_csv(DATA_PATH)

print("\n[STEP 1] DATA LOADED")
print(f"[INFO] Raw dataframe shape: {df.shape}")
print(f"[INFO] Columns available   : {len(df.columns)}")

# ------------------------------------------------------------
# FEATURE GROUPS (same spirit as v5 / v6)
# ------------------------------------------------------------
knn_feature_cols = [
    "dist_nearest_labeled_city",
    "log1p_dist_nearest_labeled_city",
    "crime_nearest_labeled_city",
    "safety_nearest_labeled_city",
    "same_country_as_nearest_labeled",
    "avg_crime_k5",
    "avg_safety_k5",
    "avg_crime_k10",
    "avg_safety_k10",
    "wavg_crime_k5",
    "wavg_safety_k5",
    "log1p_num_labeled_within_50km",
    "log1p_num_labeled_within_100km",
    "log1p_num_labeled_within_250km",
    "avg_crime_same_country_k5",
    "avg_safety_same_country_k5",
    "log1p_num_same_country_within_250km",
]

density_gravity_feature_cols = [
    "log1p_num_cities_50km",
    "log1p_sum_pop_50km",
    "log1p_pop_gravity_50km",
    "log1p_num_cities_100km",
    "log1p_sum_pop_100km",
    "log1p_pop_gravity_100km",
    "dist_to_nearest_large_city",
    "log1p_dist_to_nearest_large_city",
    "log1p_pop_of_nearest_large_city",
]

base_feature_cols = [
    "lat",
    "lon",
    "crimeindex_2020",
    "crimeindex_2023",
    "safetyindex_2020",
    "age_0_14",
    "age_15_64",
    "age_65_plus",
    "population",
    "density_per_km2",
]

macro_cols_v5 = [
    "gdp",
    "gdp_per_capita",
    "unemployment",
    "homicide_rate",
    "life_expectancy_male",
    "life_expectancy_female",
    "infant_mortality",
    "urban_population_growth",
    "tourists",
]

feature_cols_all = list(dict.fromkeys(
    knn_feature_cols + density_gravity_feature_cols + base_feature_cols + macro_cols_v5
))
feature_cols_all = [c for c in feature_cols_all if c in df.columns]

print("\n[STEP 2] FINAL AVAILABLE FEATURES")
print(f"[INFO] Number of usable model features: {len(feature_cols_all)}")
print(feature_cols_all)

# ------------------------------------------------------------
# CLEAN DATA
# ------------------------------------------------------------
if TARGET_COL not in df.columns:
    raise ValueError(f"Target column '{TARGET_COL}' not found in dataframe.")

df = df.copy()
df = df.dropna(subset=[TARGET_COL]).reset_index(drop=True)

for col in feature_cols_all:
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].median())

print("\n[STEP 3] DATA CLEANING COMPLETE")
print(f"[INFO] Final dataframe shape after target cleanup: {df.shape}")
print(f"[INFO] Remaining NaNs in features: {df[feature_cols_all].isna().sum().sum()}")

# ------------------------------------------------------------
# TRAIN / TEST SPLIT
# ------------------------------------------------------------
X = df[feature_cols_all].copy()
y = df[TARGET_COL].copy()

id_cols = [c for c in ["city", "country"] if c in df.columns]

X_train_full, X_test_full, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, df.index,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

train_df = df.loc[idx_train].reset_index(drop=True)
test_df = df.loc[idx_test].reset_index(drop=True)

X_train_full = train_df[feature_cols_all].copy()
X_test_full = test_df[feature_cols_all].copy()
y_train = train_df[TARGET_COL].copy()
y_test = test_df[TARGET_COL].copy()

print("\n[STEP 4] TRAIN / TEST SPLIT COMPLETE")
print(f"[INFO] Train shape: {X_train_full.shape}")
print(f"[INFO] Test shape : {X_test_full.shape}")

# ------------------------------------------------------------
# FEATURE SETS
# Keep this smaller than the larger search notebook
# ------------------------------------------------------------
feature_sets = {
    "geo_only": [c for c in ["lat", "lon"] if c in feature_cols_all],

    "geo_macro": [c for c in ["lat", "lon"] + macro_cols_v5 if c in feature_cols_all],

    "geo_knn_density": [
        c for c in ["lat", "lon"] + knn_feature_cols + density_gravity_feature_cols
        if c in feature_cols_all
    ],

    "all_no_current_crime": [
        c for c in feature_cols_all if c not in {"crimeindex_2023"}
    ],

    "all_features": feature_cols_all.copy(),
}

leakage_flag_cols = {"crimeindex_2023"}

print("\n[STEP 5] FEATURE SETS TO TEST")
for name, cols in feature_sets.items():
    leak_flag = int(any(c in leakage_flag_cols for c in cols))
    print(f"[FEATURE SET] {name:<22} | n_features={len(cols):<3} | uses_current_crime={leak_flag}")

feature_set_df = pd.DataFrame([
    {
        "feature_set_name": name,
        "n_features": len(cols),
        "uses_current_city_crime_signal": int(any(c in leakage_flag_cols for c in cols)),
        "feature_list": " | ".join(cols),
    }
    for name, cols in feature_sets.items()
])
feature_set_df.to_csv(ARTIFACT_DIR / "feature_set_catalog.csv", index=False)

# ------------------------------------------------------------
# METRICS
# ------------------------------------------------------------
def calc_rmse(y_true, y_pred):
    return math.sqrt(mean_squared_error(y_true, y_pred))

def score_predictions(y_true, y_pred):
    return {
        "mae": mean_absolute_error(y_true, y_pred),
        "rmse": calc_rmse(y_true, y_pred),
        "r2": r2_score(y_true, y_pred),
    }

# ------------------------------------------------------------
# MODEL BUILDERS
# ------------------------------------------------------------
def build_rf(params):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestRegressor(
            random_state=RANDOM_STATE,
            n_jobs=-1,
            **params
        ))
    ])

def build_mlp(params):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", MLPRegressor(
            random_state=RANDOM_STATE,
            early_stopping=True,
            validation_fraction=0.10,
            n_iter_no_change=40,
            max_iter=2500,
            **params
        ))
    ])

# ------------------------------------------------------------
# SMALLER HYPERPARAMETER SEARCH
# ------------------------------------------------------------
rf_grid = list(ParameterGrid({
    "n_estimators": [300, 500],
    "max_depth": [None, 16],
    "min_samples_split": [2, 4],
    "min_samples_leaf": [1, 2],
    "max_features": ["sqrt"],
}))

mlp_grid = list(ParameterGrid({
    "hidden_layer_sizes": [
        (128, 64, 32),
        (128, 128),
        (256, 128),
        (128, 64),
    ],
    "alpha": [0.0001, 0.0005],
    "learning_rate_init": [0.001, 0.0007],
    "batch_size": [64, 128],
}))

print("\n[STEP 6] MODEL GRIDS")
print(f"[INFO] RF configs : {len(rf_grid)}")
print(f"[INFO] MLP configs: {len(mlp_grid)}")
print(f"[INFO] Feature sets: {len(feature_sets)}")
print(f"[INFO] Total models to train: {len(feature_sets) * (len(rf_grid) + len(mlp_grid))}")

# ------------------------------------------------------------
# CV SETUP
# ------------------------------------------------------------
cv = KFold(n_splits=N_SPLITS_CV, shuffle=True, random_state=RANDOM_STATE)

# ------------------------------------------------------------
# TRAIN LOOP
# ------------------------------------------------------------
all_results = []
best_model_bundle = None
best_test_rmse = float("inf")

run_id = 0

for feature_set_name, feature_list in feature_sets.items():
    X_train_fs = X_train_full[feature_list].copy()
    X_test_fs = X_test_full[feature_list].copy()
    uses_current_crime = int(any(c in leakage_flag_cols for c in feature_list))

    # ---------------- RF ----------------
    for params in rf_grid:
        run_id += 1
        model_name = "RandomForestRegressor"
        run_label = f"run_{run_id:03d}_rf_{feature_set_name}"

        print("\n" + "-" * 100)
        print(f"[RUN {run_id:03d}] {model_name}")
        print(f"[INFO] Feature set : {feature_set_name}")
        print(f"[INFO] # Features  : {len(feature_list)}")
        print(f"[INFO] Params      : {json.dumps(params)}")

        cv_rows = []
        for fold_num, (tr_idx, val_idx) in enumerate(cv.split(X_train_fs, y_train), start=1):
            X_tr = X_train_fs.iloc[tr_idx]
            X_val = X_train_fs.iloc[val_idx]
            y_tr = y_train.iloc[tr_idx]
            y_val = y_train.iloc[val_idx]

            model = build_rf(params)
            model.fit(X_tr, y_tr)
            val_pred = model.predict(X_val)
            val_scores = score_predictions(y_val, val_pred)
            cv_rows.append(val_scores)

            print(
                f"[CV] Fold {fold_num}/{N_SPLITS_CV} | "
                f"MAE={val_scores['mae']:.4f} | "
                f"RMSE={val_scores['rmse']:.4f} | "
                f"R2={val_scores['r2']:.4f}"
            )

        final_model = build_rf(params)
        final_model.fit(X_train_fs, y_train)

        train_pred = final_model.predict(X_train_fs)
        test_pred = final_model.predict(X_test_fs)

        train_scores = score_predictions(y_train, train_pred)
        test_scores = score_predictions(y_test, test_pred)

        result_row = {
            "run_id": run_id,
            "model_family": "rf",
            "model_name": model_name,
            "feature_set_name": feature_set_name,
            "n_features": len(feature_list),
            "uses_current_city_crime_signal": uses_current_crime,
            "cv_mae_mean": np.mean([x["mae"] for x in cv_rows]),
            "cv_rmse_mean": np.mean([x["rmse"] for x in cv_rows]),
            "cv_r2_mean": np.mean([x["r2"] for x in cv_rows]),
            "cv_rmse_std": np.std([x["rmse"] for x in cv_rows]),
            "train_mae": train_scores["mae"],
            "train_rmse": train_scores["rmse"],
            "train_r2": train_scores["r2"],
            "test_mae": test_scores["mae"],
            "test_rmse": test_scores["rmse"],
            "test_r2": test_scores["r2"],
            "rmse_overfit_gap": test_scores["rmse"] - train_scores["rmse"],
            "params_json": json.dumps(params, sort_keys=True),
            "feature_list": " | ".join(feature_list),
        }
        all_results.append(result_row)

        print(f"[RESULT] Train RMSE : {train_scores['rmse']:.4f}")
        print(f"[RESULT] Test RMSE  : {test_scores['rmse']:.4f}")
        print(f"[RESULT] Test MAE   : {test_scores['mae']:.4f}")
        print(f"[RESULT] Test R2    : {test_scores['r2']:.4f}")

        if test_scores["rmse"] < best_test_rmse:
            best_test_rmse = test_scores["rmse"]
            best_model_bundle = {
                "run_row": result_row,
                "model": clone(final_model),
                "feature_list": feature_list.copy(),
                "test_pred": test_pred.copy(),
                "train_pred": train_pred.copy(),
            }
            best_model_bundle["model"].fit(X_train_fs, y_train)

            print(f"[BEST SO FAR] New best test RMSE: {best_test_rmse:.4f}")

    # ---------------- MLP ----------------
    for params in mlp_grid:
        run_id += 1
        model_name = "MLPRegressor"
        run_label = f"run_{run_id:03d}_mlp_{feature_set_name}"

        print("\n" + "-" * 100)
        print(f"[RUN {run_id:03d}] {model_name}")
        print(f"[INFO] Feature set : {feature_set_name}")
        print(f"[INFO] # Features  : {len(feature_list)}")
        print(f"[INFO] Params      : {json.dumps(params, default=str)}")

        cv_rows = []
        for fold_num, (tr_idx, val_idx) in enumerate(cv.split(X_train_fs, y_train), start=1):
            X_tr = X_train_fs.iloc[tr_idx]
            X_val = X_train_fs.iloc[val_idx]
            y_tr = y_train.iloc[tr_idx]
            y_val = y_train.iloc[val_idx]

            model = build_mlp(params)
            model.fit(X_tr, y_tr)
            val_pred = model.predict(X_val)
            val_scores = score_predictions(y_val, val_pred)
            cv_rows.append(val_scores)

            print(
                f"[CV] Fold {fold_num}/{N_SPLITS_CV} | "
                f"MAE={val_scores['mae']:.4f} | "
                f"RMSE={val_scores['rmse']:.4f} | "
                f"R2={val_scores['r2']:.4f}"
            )

        final_model = build_mlp(params)
        final_model.fit(X_train_fs, y_train)

        train_pred = final_model.predict(X_train_fs)
        test_pred = final_model.predict(X_test_fs)

        train_scores = score_predictions(y_train, train_pred)
        test_scores = score_predictions(y_test, test_pred)

        fitted_mlp = final_model.named_steps["model"]
        n_iter_used = getattr(fitted_mlp, "n_iter_", np.nan)

        result_row = {
            "run_id": run_id,
            "model_family": "mlp",
            "model_name": model_name,
            "feature_set_name": feature_set_name,
            "n_features": len(feature_list),
            "uses_current_city_crime_signal": uses_current_crime,
            "cv_mae_mean": np.mean([x["mae"] for x in cv_rows]),
            "cv_rmse_mean": np.mean([x["rmse"] for x in cv_rows]),
            "cv_r2_mean": np.mean([x["r2"] for x in cv_rows]),
            "cv_rmse_std": np.std([x["rmse"] for x in cv_rows]),
            "train_mae": train_scores["mae"],
            "train_rmse": train_scores["rmse"],
            "train_r2": train_scores["r2"],
            "test_mae": test_scores["mae"],
            "test_rmse": test_scores["rmse"],
            "test_r2": test_scores["r2"],
            "rmse_overfit_gap": test_scores["rmse"] - train_scores["rmse"],
            "mlp_n_iter": n_iter_used,
            "params_json": json.dumps(params, default=str, sort_keys=True),
            "feature_list": " | ".join(feature_list),
        }
        all_results.append(result_row)

        print(f"[RESULT] Train RMSE : {train_scores['rmse']:.4f}")
        print(f"[RESULT] Test RMSE  : {test_scores['rmse']:.4f}")
        print(f"[RESULT] Test MAE   : {test_scores['mae']:.4f}")
        print(f"[RESULT] Test R2    : {test_scores['r2']:.4f}")
        print(f"[RESULT] MLP iters  : {n_iter_used}")

        if test_scores["rmse"] < best_test_rmse:
            best_test_rmse = test_scores["rmse"]
            best_model_bundle = {
                "run_row": result_row,
                "model": clone(final_model),
                "feature_list": feature_list.copy(),
                "test_pred": test_pred.copy(),
                "train_pred": train_pred.copy(),
            }
            best_model_bundle["model"].fit(X_train_fs, y_train)

            print(f"[BEST SO FAR] New best test RMSE: {best_test_rmse:.4f}")

# ------------------------------------------------------------
# RESULTS DATAFRAME
# ------------------------------------------------------------
results_df = pd.DataFrame(all_results).sort_values(
    ["test_rmse", "cv_rmse_mean", "cv_rmse_std"],
    ascending=[True, True, True]
).reset_index(drop=True)

results_csv_path = ARTIFACT_DIR / "rf_mlp_model_search_results.csv"
results_df.to_csv(results_csv_path, index=False)

print("\n" + "=" * 100)
print("[STEP 7] ALL MODEL RUNS COMPLETE")
print(f"[INFO] Total models trained: {len(results_df)}")
print("[INFO] Top 20 runs by test RMSE:")
print(results_df.head(20)[[
    "run_id",
    "model_family",
    "feature_set_name",
    "n_features",
    "uses_current_city_crime_signal",
    "cv_rmse_mean",
    "cv_rmse_std",
    "test_rmse",
    "test_mae",
    "test_r2",
    "rmse_overfit_gap"
]].to_string(index=False))

# ------------------------------------------------------------
# SEPARATE LEADERBOARDS
# ------------------------------------------------------------
leaderboard_no_current_crime = results_df[
    results_df["uses_current_city_crime_signal"] == 0
].reset_index(drop=True)

leaderboard_with_current_crime = results_df[
    results_df["uses_current_city_crime_signal"] == 1
].reset_index(drop=True)

leaderboard_no_current_crime.to_csv(
    ARTIFACT_DIR / "leaderboard_no_current_city_crime_signal.csv",
    index=False
)

leaderboard_with_current_crime.to_csv(
    ARTIFACT_DIR / "leaderboard_with_current_city_crime_signal.csv",
    index=False
)

print("\n[STEP 8] LEADERBOARDS SAVED")
if len(leaderboard_no_current_crime) > 0:
    print("[BEST NO CURRENT CRIME]")
    print(leaderboard_no_current_crime.head(5)[[
        "run_id", "model_family", "feature_set_name", "test_rmse", "test_mae", "test_r2"
    ]].to_string(index=False))

if len(leaderboard_with_current_crime) > 0:
    print("\n[BEST WITH CURRENT CRIME]")
    print(leaderboard_with_current_crime.head(5)[[
        "run_id", "model_family", "feature_set_name", "test_rmse", "test_mae", "test_r2"
    ]].to_string(index=False))

# ------------------------------------------------------------
# SAVE BEST MODEL + PREDICTIONS
# ------------------------------------------------------------
if best_model_bundle is None:
    raise RuntimeError("No best model found.")

best_row = best_model_bundle["run_row"]
best_model = best_model_bundle["model"]
best_features = best_model_bundle["feature_list"]

joblib.dump(best_model, ARTIFACT_DIR / "best_model.joblib")
joblib.dump(best_features, ARTIFACT_DIR / "best_model_feature_list.joblib")

prediction_table = test_df[id_cols + [TARGET_COL]].copy() if len(id_cols) > 0 else test_df[[TARGET_COL]].copy()
prediction_table["best_model_prediction"] = best_model_bundle["test_pred"]
prediction_table["absolute_error"] = (
    prediction_table[TARGET_COL] - prediction_table["best_model_prediction"]
).abs()

prediction_table = prediction_table.sort_values("absolute_error").reset_index(drop=True)
prediction_table.to_csv(ARTIFACT_DIR / "best_model_test_predictions.csv", index=False)

summary_rows = []

summary_rows.append({
    "summary_type": "best_overall",
    **best_row
})

if len(leaderboard_no_current_crime) > 0:
    summary_rows.append({
        "summary_type": "best_no_current_city_crime_signal",
        **leaderboard_no_current_crime.iloc[0].to_dict()
    })

if len(leaderboard_with_current_crime) > 0:
    summary_rows.append({
        "summary_type": "best_with_current_city_crime_signal",
        **leaderboard_with_current_crime.iloc[0].to_dict()
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(ARTIFACT_DIR / "best_model_summary.csv", index=False)

print("\n" + "=" * 100)
print("[STEP 9] BEST MODEL SUMMARY")
print("=" * 100)
for k, v in best_row.items():
    print(f"{k}: {v}")

print("\n[FILES SAVED]")
print(ARTIFACT_DIR / "feature_set_catalog.csv")
print(ARTIFACT_DIR / "rf_mlp_model_search_results.csv")
print(ARTIFACT_DIR / "leaderboard_no_current_city_crime_signal.csv")
print(ARTIFACT_DIR / "leaderboard_with_current_city_crime_signal.csv")
print(ARTIFACT_DIR / "best_model.joblib")
print(ARTIFACT_DIR / "best_model_feature_list.joblib")
print(ARTIFACT_DIR / "best_model_test_predictions.csv")
print(ARTIFACT_DIR / "best_model_summary.csv")
print("=" * 100)

GEO SAFETY RF + MLP LIGHT SEARCH
[CONFIG] DATA_PATH      : data/compiled_model_ready/MR_cities_worldpop_knn_macro_v5.csv
[CONFIG] TARGET_COL     : safety_index
[CONFIG] TEST_SIZE      : 0.2
[CONFIG] CV_FOLDS       : 5
[CONFIG] OUTPUT_DIR     : artifacts/geo_safety_rf_mlp_light

[STEP 1] DATA LOADED
[INFO] Raw dataframe shape: (509, 77)
[INFO] Columns available   : 77

[STEP 2] FINAL AVAILABLE FEATURES
[INFO] Number of usable model features: 45
['dist_nearest_labeled_city', 'log1p_dist_nearest_labeled_city', 'crime_nearest_labeled_city', 'safety_nearest_labeled_city', 'same_country_as_nearest_labeled', 'avg_crime_k5', 'avg_safety_k5', 'avg_crime_k10', 'avg_safety_k10', 'wavg_crime_k5', 'wavg_safety_k5', 'log1p_num_labeled_within_50km', 'log1p_num_labeled_within_100km', 'log1p_num_labeled_within_250km', 'avg_crime_same_country_k5', 'avg_safety_same_country_k5', 'log1p_num_same_country_within_250km', 'log1p_num_cities_50km', 'log1p_sum_pop_50km', 'log1p_pop_gravity_50km', 'log1p_num_citie

In [4]:
# ============================================================
# GEO SAFETY MODEL SEARCH - BAYESIAN / ADAPTIVE VERSION
# Optuna + RandomForestRegressor + sklearn MLPRegressor
# Dynamic feature-set + hyperparameter search
# ============================================================

import os
import json
import math
import subprocess
import sys
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# OPTIONAL: INSTALL OPTUNA IF NEEDED
# ------------------------------------------------------------
try:
    import optuna
    from optuna.samplers import TPESampler
    from optuna.pruners import MedianPruner
except ImportError:
    print("[SETUP] Optuna not found. Installing now...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "optuna", "-q"])
    import optuna
    from optuna.samplers import TPESampler
    from optuna.pruners import MedianPruner

# ------------------------------------------------------------
# NOTEBOOK OUTPUT: FORCE NON-SCROLLING OUTPUT
# ------------------------------------------------------------
try:
    from IPython.display import HTML, display
    display(HTML("""
    <style>
    .output_scroll, .jp-OutputArea-output, .jp-OutputArea-child {
        height: auto !important;
        max-height: none !important;
        overflow-y: visible !important;
    }
    </style>
    """))
except Exception:
    pass

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
RANDOM_STATE = 42
TEST_SIZE = 0.20
N_SPLITS_CV = 5
N_TRIALS = 120                   # start here; raise later if needed
TIMEOUT_SECONDS = None           # e.g. 3600 for one hour cap
OPTUNA_STUDY_NAME = "geo_safety_rf_mlp_bayesian_v1"

# >>> CHANGE THIS IF NEEDED <<<
DATA_PATH = Path("data/compiled_model_ready/MR_cities_worldpop_knn_macro_v5.csv")

ARTIFACT_DIR = Path("artifacts/geo_safety_optuna_rf_mlp")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COL = "safety_index"

print("=" * 110)
print("GEO SAFETY - DYNAMIC BAYESIAN SEARCH (RF + MLP)")
print("=" * 110)
print(f"[CONFIG] DATA_PATH        : {DATA_PATH}")
print(f"[CONFIG] TARGET_COL       : {TARGET_COL}")
print(f"[CONFIG] TEST_SIZE        : {TEST_SIZE}")
print(f"[CONFIG] CV_FOLDS         : {N_SPLITS_CV}")
print(f"[CONFIG] N_TRIALS         : {N_TRIALS}")
print(f"[CONFIG] TIMEOUT_SECONDS  : {TIMEOUT_SECONDS}")
print(f"[CONFIG] OUTPUT_DIR       : {ARTIFACT_DIR}")
print("=" * 110)

# ------------------------------------------------------------
# LOAD DATA
# ------------------------------------------------------------
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

df = pd.read_csv(DATA_PATH)

print("\n[STEP 1] DATA LOADED")
print(f"[INFO] Raw dataframe shape: {df.shape}")
print(f"[INFO] Total columns      : {len(df.columns)}")

# ------------------------------------------------------------
# FEATURE GROUPS ALIGNED TO v5 / v6 NOTEBOOKS
# ------------------------------------------------------------
knn_feature_cols = [
    "dist_nearest_labeled_city",
    "log1p_dist_nearest_labeled_city",
    "crime_nearest_labeled_city",
    "safety_nearest_labeled_city",
    "same_country_as_nearest_labeled",
    "avg_crime_k5",
    "avg_safety_k5",
    "avg_crime_k10",
    "avg_safety_k10",
    "wavg_crime_k5",
    "wavg_safety_k5",
    "log1p_num_labeled_within_50km",
    "log1p_num_labeled_within_100km",
    "log1p_num_labeled_within_250km",
    "avg_crime_same_country_k5",
    "avg_safety_same_country_k5",
    "log1p_num_same_country_within_250km",
]

density_gravity_feature_cols = [
    "log1p_num_cities_50km",
    "log1p_sum_pop_50km",
    "log1p_pop_gravity_50km",
    "log1p_num_cities_100km",
    "log1p_sum_pop_100km",
    "log1p_pop_gravity_100km",
    "dist_to_nearest_large_city",
    "log1p_dist_to_nearest_large_city",
    "log1p_pop_of_nearest_large_city",
]

base_feature_cols = [
    "lat",
    "lon",
    "crimeindex_2020",
    "crimeindex_2023",
    "safetyindex_2020",
    "age_0_14",
    "age_15_64",
    "age_65_plus",
    "population",
    "density_per_km2",
]

macro_cols_v5 = [
    "gdp",
    "gdp_per_capita",
    "unemployment",
    "homicide_rate",
    "life_expectancy_male",
    "life_expectancy_female",
    "infant_mortality",
    "urban_population_growth",
    "tourists",
]

feature_cols_all = list(dict.fromkeys(
    knn_feature_cols + density_gravity_feature_cols + base_feature_cols + macro_cols_v5
))
feature_cols_all = [c for c in feature_cols_all if c in df.columns]

print("\n[STEP 2] AVAILABLE MODEL FEATURES")
print(f"[INFO] Usable feature count: {len(feature_cols_all)}")
print(feature_cols_all)

# ------------------------------------------------------------
# CLEAN / IMPUTE
# ------------------------------------------------------------
if TARGET_COL not in df.columns:
    raise ValueError(f"Target column '{TARGET_COL}' not found.")

df = df.copy()
df = df.dropna(subset=[TARGET_COL]).reset_index(drop=True)

for col in feature_cols_all:
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].median())

print("\n[STEP 3] DATA CLEANING COMPLETE")
print(f"[INFO] Final dataframe shape : {df.shape}")
print(f"[INFO] Remaining feature NaNs: {df[feature_cols_all].isna().sum().sum()}")

# ------------------------------------------------------------
# TRAIN / TEST SPLIT
# IMPORTANT: TEST SET IS HELD OUT FROM TUNING
# ------------------------------------------------------------
X_all = df[feature_cols_all].copy()
y_all = df[TARGET_COL].astype(float).copy()

X_train_full, X_test_full, y_train, y_test, idx_train, idx_test = train_test_split(
    X_all, y_all, df.index,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

train_df = df.loc[idx_train].reset_index(drop=True)
test_df = df.loc[idx_test].reset_index(drop=True)

X_train_full = train_df[feature_cols_all].copy()
X_test_full = test_df[feature_cols_all].copy()
y_train = train_df[TARGET_COL].astype(float).copy()
y_test = test_df[TARGET_COL].astype(float).copy()

id_cols = [c for c in ["city", "country"] if c in df.columns]

print("\n[STEP 4] HOLDOUT SPLIT CREATED")
print(f"[INFO] Train shape: {X_train_full.shape}")
print(f"[INFO] Test shape : {X_test_full.shape}")

# ------------------------------------------------------------
# FEATURE SET OPTIONS
# Keep compact but meaningful
# ------------------------------------------------------------
feature_sets = {
    "geo_only": [c for c in ["lat", "lon"] if c in feature_cols_all],

    "geo_macro": [c for c in ["lat", "lon"] + macro_cols_v5 if c in feature_cols_all],

    "geo_knn_density": [
        c for c in ["lat", "lon"] + knn_feature_cols + density_gravity_feature_cols
        if c in feature_cols_all
    ],

    "all_no_current_crime": [
        c for c in feature_cols_all if c not in {"crimeindex_2023"}
    ],

    "all_features": feature_cols_all.copy(),
}

feature_set_names = list(feature_sets.keys())
current_city_crime_like_cols = {"crimeindex_2023"}

print("\n[STEP 5] FEATURE SETS AVAILABLE TO BAYESIAN SEARCH")
for fs_name, fs_cols in feature_sets.items():
    uses_current_crime = int(any(c in current_city_crime_like_cols for c in fs_cols))
    print(f"[FEATURE SET] {fs_name:<22} | n_features={len(fs_cols):<3} | uses_current_crime={uses_current_crime}")

pd.DataFrame([
    {
        "feature_set_name": name,
        "n_features": len(cols),
        "uses_current_city_crime_signal": int(any(c in current_city_crime_like_cols for c in cols)),
        "feature_list": " | ".join(cols),
    }
    for name, cols in feature_sets.items()
]).to_csv(ARTIFACT_DIR / "feature_set_catalog.csv", index=False)

# ------------------------------------------------------------
# METRICS
# ------------------------------------------------------------
def calc_rmse(y_true, y_pred):
    return math.sqrt(mean_squared_error(y_true, y_pred))

def score_predictions(y_true, y_pred):
    return {
        "mae": mean_absolute_error(y_true, y_pred),
        "rmse": calc_rmse(y_true, y_pred),
        "r2": r2_score(y_true, y_pred),
    }

# ------------------------------------------------------------
# MODEL BUILDERS
# ------------------------------------------------------------
def build_rf(params):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestRegressor(
            random_state=RANDOM_STATE,
            n_jobs=-1,
            **params
        ))
    ])

def build_mlp(params):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", MLPRegressor(
            random_state=RANDOM_STATE,
            solver="adam",
            early_stopping=True,
            validation_fraction=0.10,
            n_iter_no_change=40,
            max_iter=2500,
            **params
        ))
    ])

# ------------------------------------------------------------
# CROSS-VALIDATION
# ------------------------------------------------------------
cv = KFold(n_splits=N_SPLITS_CV, shuffle=True, random_state=RANDOM_STATE)

# ------------------------------------------------------------
# GLOBAL TRIAL STORE
# ------------------------------------------------------------
trial_rows = []

# ------------------------------------------------------------
# OPTUNA OBJECTIVE
# direction = minimize CV RMSE
# ------------------------------------------------------------
def objective(trial):
    model_family = trial.suggest_categorical("model_family", ["rf", "mlp"])
    feature_set_name = trial.suggest_categorical("feature_set_name", feature_set_names)
    feature_list = feature_sets[feature_set_name]

    X_train_fs = X_train_full[feature_list].copy()
    uses_current_crime = int(any(c in current_city_crime_like_cols for c in feature_list))

    if model_family == "rf":
        params = {
            "n_estimators": trial.suggest_int("rf_n_estimators", 200, 900, step=100),
            "max_depth": trial.suggest_categorical("rf_max_depth", [None, 8, 12, 16, 20, 30]),
            "min_samples_split": trial.suggest_int("rf_min_samples_split", 2, 10),
            "min_samples_leaf": trial.suggest_int("rf_min_samples_leaf", 1, 5),
            "max_features": trial.suggest_categorical("rf_max_features", ["sqrt", "log2", 0.7, 0.9]),
            "bootstrap": trial.suggest_categorical("rf_bootstrap", [True, False]),
        }
        model = build_rf(params)

    else:
        hidden_choice = trial.suggest_categorical(
            "mlp_hidden_layer_sizes",
            [
                (64,),
                (128,),
                (256,),
                (128, 64),
                (128, 128),
                (256, 128),
                (128, 64, 32),
                (256, 128, 64),
            ]
        )

        params = {
            "hidden_layer_sizes": hidden_choice,
            "activation": trial.suggest_categorical("mlp_activation", ["relu", "tanh"]),
            "alpha": trial.suggest_float("mlp_alpha", 1e-5, 5e-3, log=True),
            "learning_rate_init": trial.suggest_float("mlp_learning_rate_init", 2e-4, 2e-3, log=True),
            "batch_size": trial.suggest_categorical("mlp_batch_size", [32, 64, 128, 256]),
            "tol": trial.suggest_float("mlp_tol", 1e-5, 1e-3, log=True),
        }
        model = build_mlp(params)

    print("\n" + "-" * 110)
    print(f"[TRIAL {trial.number:03d}] START")
    print(f"[INFO] Model family : {model_family}")
    print(f"[INFO] Feature set  : {feature_set_name}")
    print(f"[INFO] # Features   : {len(feature_list)}")
    print(f"[INFO] Params       : {json.dumps(params, default=str)}")

    fold_metrics = []
    for fold_idx, (tr_idx, val_idx) in enumerate(cv.split(X_train_fs, y_train), start=1):
        X_tr = X_train_fs.iloc[tr_idx]
        X_val = X_train_fs.iloc[val_idx]
        y_tr = y_train.iloc[tr_idx]
        y_val = y_train.iloc[val_idx]

        fold_model = clone(model)
        fold_model.fit(X_tr, y_tr)
        val_pred = fold_model.predict(X_val)
        fold_scores = score_predictions(y_val, val_pred)
        fold_metrics.append(fold_scores)

        current_mean_rmse = float(np.mean([m["rmse"] for m in fold_metrics]))
        trial.report(current_mean_rmse, step=fold_idx)

        print(
            f"[CV] Fold {fold_idx}/{N_SPLITS_CV} | "
            f"MAE={fold_scores['mae']:.4f} | "
            f"RMSE={fold_scores['rmse']:.4f} | "
            f"R2={fold_scores['r2']:.4f} | "
            f"Running mean RMSE={current_mean_rmse:.4f}"
        )

        if trial.should_prune():
            print(f"[TRIAL {trial.number:03d}] PRUNED after fold {fold_idx}")
            raise optuna.TrialPruned()

    cv_mae_mean = float(np.mean([m["mae"] for m in fold_metrics]))
    cv_rmse_mean = float(np.mean([m["rmse"] for m in fold_metrics]))
    cv_r2_mean = float(np.mean([m["r2"] for m in fold_metrics]))
    cv_rmse_std = float(np.std([m["rmse"] for m in fold_metrics]))

    trial.set_user_attr("model_family", model_family)
    trial.set_user_attr("feature_set_name", feature_set_name)
    trial.set_user_attr("n_features", len(feature_list))
    trial.set_user_attr("uses_current_city_crime_signal", uses_current_crime)
    trial.set_user_attr("cv_mae_mean", cv_mae_mean)
    trial.set_user_attr("cv_rmse_mean", cv_rmse_mean)
    trial.set_user_attr("cv_r2_mean", cv_r2_mean)
    trial.set_user_attr("cv_rmse_std", cv_rmse_std)
    trial.set_user_attr("feature_list", " | ".join(feature_list))
    trial.set_user_attr("params_json", json.dumps(params, default=str, sort_keys=True))

    trial_rows.append({
        "trial_number": trial.number,
        "state": "COMPLETE",
        "model_family": model_family,
        "feature_set_name": feature_set_name,
        "n_features": len(feature_list),
        "uses_current_city_crime_signal": uses_current_crime,
        "cv_mae_mean": cv_mae_mean,
        "cv_rmse_mean": cv_rmse_mean,
        "cv_r2_mean": cv_r2_mean,
        "cv_rmse_std": cv_rmse_std,
        "params_json": json.dumps(params, default=str, sort_keys=True),
        "feature_list": " | ".join(feature_list),
    })

    print(f"[TRIAL {trial.number:03d}] COMPLETE")
    print(f"[RESULT] CV mean RMSE: {cv_rmse_mean:.4f}")
    print(f"[RESULT] CV std RMSE : {cv_rmse_std:.4f}")
    print(f"[RESULT] CV mean MAE : {cv_mae_mean:.4f}")
    print(f"[RESULT] CV mean R2  : {cv_r2_mean:.4f}")

    # Save incremental log after each successful trial
    pd.DataFrame(trial_rows).to_csv(ARTIFACT_DIR / "optuna_trial_log_live.csv", index=False)

    return cv_rmse_mean

# ------------------------------------------------------------
# CALLBACK FOR LOGGING
# ------------------------------------------------------------
def logging_callback(study, trial):
    print("\n" + "=" * 110)
    print(f"[STUDY UPDATE] Finished trial {trial.number}")
    print(f"[STUDY UPDATE] Trial state : {trial.state}")
    if trial.state.name == "COMPLETE":
        print(f"[STUDY UPDATE] Trial value : {trial.value:.4f}")
    if study.best_trial is not None:
        print(f"[STUDY UPDATE] Best trial so far : {study.best_trial.number}")
        print(f"[STUDY UPDATE] Best CV RMSE      : {study.best_value:.4f}")
        print(f"[STUDY UPDATE] Best params       : {study.best_trial.params}")
    print("=" * 110)

# ------------------------------------------------------------
# RUN OPTUNA STUDY
# TPE is a Bayesian-style adaptive sampler in Optuna
# MedianPruner can stop weak trials early
# ------------------------------------------------------------
sampler = TPESampler(seed=RANDOM_STATE)
pruner = MedianPruner(n_startup_trials=10, n_warmup_steps=2)

study = optuna.create_study(
    study_name=OPTUNA_STUDY_NAME,
    direction="minimize",
    sampler=sampler,
    pruner=pruner,
)

print("\n[STEP 6] STARTING OPTUNA BAYESIAN / ADAPTIVE SEARCH")
study.optimize(
    objective,
    n_trials=N_TRIALS,
    timeout=TIMEOUT_SECONDS,
    callbacks=[logging_callback],
    show_progress_bar=False
)

# ------------------------------------------------------------
# EXPORT OPTUNA TRIALS
# ------------------------------------------------------------
print("\n[STEP 7] EXPORTING STUDY RESULTS")

trials_df = study.trials_dataframe()
trials_df.to_csv(ARTIFACT_DIR / "optuna_trials_dataframe.csv", index=False)

if len(trial_rows) > 0:
    live_trials_df = pd.DataFrame(trial_rows).sort_values("cv_rmse_mean").reset_index(drop=True)
else:
    live_trials_df = pd.DataFrame(columns=[
        "trial_number", "state", "model_family", "feature_set_name", "n_features",
        "uses_current_city_crime_signal", "cv_mae_mean", "cv_rmse_mean", "cv_r2_mean",
        "cv_rmse_std", "params_json", "feature_list"
    ])

live_trials_df.to_csv(ARTIFACT_DIR / "optuna_trial_log_final.csv", index=False)

print(f"[INFO] Total completed logged trials: {len(live_trials_df)}")
if len(live_trials_df) > 0:
    print("[INFO] Top 15 completed trials by CV RMSE:")
    print(live_trials_df.head(15).to_string(index=False))

# ------------------------------------------------------------
# REFIT BEST MODEL ON FULL TRAINING SET
# THEN EVALUATE ON THE TRUE HOLDOUT TEST SET
# ------------------------------------------------------------
print("\n[STEP 8] REFITTING BEST MODEL ON FULL TRAINING SET")

best_trial = study.best_trial
best_params = best_trial.params.copy()
best_model_family = best_params["model_family"]
best_feature_set_name = best_params["feature_set_name"]
best_feature_list = feature_sets[best_feature_set_name]

def build_best_model_from_trial_params(trial_params):
    model_family = trial_params["model_family"]

    if model_family == "rf":
        rf_params = {
            "n_estimators": trial_params["rf_n_estimators"],
            "max_depth": trial_params["rf_max_depth"],
            "min_samples_split": trial_params["rf_min_samples_split"],
            "min_samples_leaf": trial_params["rf_min_samples_leaf"],
            "max_features": trial_params["rf_max_features"],
            "bootstrap": trial_params["rf_bootstrap"],
        }
        return build_rf(rf_params), rf_params

    mlp_params = {
        "hidden_layer_sizes": trial_params["mlp_hidden_layer_sizes"],
        "activation": trial_params["mlp_activation"],
        "alpha": trial_params["mlp_alpha"],
        "learning_rate_init": trial_params["mlp_learning_rate_init"],
        "batch_size": trial_params["mlp_batch_size"],
        "tol": trial_params["mlp_tol"],
    }
    return build_mlp(mlp_params), mlp_params

best_model, best_model_params = build_best_model_from_trial_params(best_params)

X_train_best = X_train_full[best_feature_list].copy()
X_test_best = X_test_full[best_feature_list].copy()

best_model.fit(X_train_best, y_train)

train_pred = best_model.predict(X_train_best)
test_pred = best_model.predict(X_test_best)

train_scores = score_predictions(y_train, train_pred)
test_scores = score_predictions(y_test, test_pred)

print("\n" + "=" * 110)
print("BEST FINAL MODEL")
print("=" * 110)
print(f"[BEST] Model family        : {best_model_family}")
print(f"[BEST] Feature set         : {best_feature_set_name}")
print(f"[BEST] Number of features  : {len(best_feature_list)}")
print(f"[BEST] Uses current crime? : {int(any(c in current_city_crime_like_cols for c in best_feature_list))}")
print(f"[BEST] Best CV RMSE        : {study.best_value:.4f}")
print(f"[BEST] Train MAE           : {train_scores['mae']:.4f}")
print(f"[BEST] Train RMSE          : {train_scores['rmse']:.4f}")
print(f"[BEST] Train R2            : {train_scores['r2']:.4f}")
print(f"[BEST] Test MAE            : {test_scores['mae']:.4f}")
print(f"[BEST] Test RMSE           : {test_scores['rmse']:.4f}")
print(f"[BEST] Test R2             : {test_scores['r2']:.4f}")
print(f"[BEST] Overfit gap RMSE    : {test_scores['rmse'] - train_scores['rmse']:.4f}")
print(f"[BEST] Params              : {json.dumps(best_model_params, default=str, indent=2)}")
print("=" * 110)

# ------------------------------------------------------------
# SAVE BEST MODEL ARTIFACTS
# ------------------------------------------------------------
joblib.dump(best_model, ARTIFACT_DIR / "best_optuna_model.joblib")
joblib.dump(best_feature_list, ARTIFACT_DIR / "best_optuna_feature_list.joblib")

best_summary_df = pd.DataFrame([{
    "best_trial_number": best_trial.number,
    "model_family": best_model_family,
    "feature_set_name": best_feature_set_name,
    "n_features": len(best_feature_list),
    "uses_current_city_crime_signal": int(any(c in current_city_crime_like_cols for c in best_feature_list)),
    "best_cv_rmse": study.best_value,
    "train_mae": train_scores["mae"],
    "train_rmse": train_scores["rmse"],
    "train_r2": train_scores["r2"],
    "test_mae": test_scores["mae"],
    "test_rmse": test_scores["rmse"],
    "test_r2": test_scores["r2"],
    "rmse_overfit_gap": test_scores["rmse"] - train_scores["rmse"],
    "best_params_json": json.dumps(best_model_params, default=str, sort_keys=True),
    "feature_list": " | ".join(best_feature_list),
}])
best_summary_df.to_csv(ARTIFACT_DIR / "best_model_summary.csv", index=False)

# ------------------------------------------------------------
# SAVE TEST PREDICTIONS
# ------------------------------------------------------------
prediction_cols = id_cols + [TARGET_COL] if len(id_cols) > 0 else [TARGET_COL]
best_pred_df = test_df[prediction_cols].copy()
best_pred_df["best_model_prediction"] = test_pred
best_pred_df["absolute_error"] = (
    best_pred_df[TARGET_COL] - best_pred_df["best_model_prediction"]
).abs()
best_pred_df = best_pred_df.sort_values("absolute_error").reset_index(drop=True)
best_pred_df.to_csv(ARTIFACT_DIR / "best_model_test_predictions.csv", index=False)

# ------------------------------------------------------------
# SAVE LEADERBOARDS
# ------------------------------------------------------------
if len(live_trials_df) > 0:
    leaderboard_all = live_trials_df.sort_values(
        ["cv_rmse_mean", "cv_rmse_std"], ascending=[True, True]
    ).reset_index(drop=True)

    leaderboard_no_current_crime = leaderboard_all[
        leaderboard_all["uses_current_city_crime_signal"] == 0
    ].reset_index(drop=True)

    leaderboard_with_current_crime = leaderboard_all[
        leaderboard_all["uses_current_city_crime_signal"] == 1
    ].reset_index(drop=True)

    leaderboard_all.to_csv(ARTIFACT_DIR / "leaderboard_all_completed_trials.csv", index=False)
    leaderboard_no_current_crime.to_csv(
        ARTIFACT_DIR / "leaderboard_no_current_city_crime_signal.csv", index=False
    )
    leaderboard_with_current_crime.to_csv(
        ARTIFACT_DIR / "leaderboard_with_current_city_crime_signal.csv", index=False
    )

    print("\n[STEP 9] LEADERBOARDS SAVED")
    print("[TOP 10 ALL COMPLETED TRIALS]")
    print(leaderboard_all.head(10)[[
        "trial_number", "model_family", "feature_set_name", "cv_rmse_mean", "cv_mae_mean", "cv_r2_mean"
    ]].to_string(index=False))

# ------------------------------------------------------------
# OPTIONAL: RANDOM FOREST FEATURE IMPORTANCE FOR BEST RF MODEL
# ------------------------------------------------------------
try:
    if best_model_family == "rf":
        rf_model = best_model.named_steps["model"]
        rf_importance_df = pd.DataFrame({
            "feature": best_feature_list,
            "importance": rf_model.feature_importances_,
        }).sort_values("importance", ascending=False).reset_index(drop=True)
        rf_importance_df.to_csv(ARTIFACT_DIR / "best_rf_feature_importance.csv", index=False)

        print("\n[INFO] Best model is RF. Top feature importances:")
        print(rf_importance_df.head(20).to_string(index=False))
except Exception as e:
    print(f"[WARNING] RF feature importance save failed: {e}")

# ------------------------------------------------------------
# FINAL FILE LIST
# ------------------------------------------------------------
print("\n" + "=" * 110)
print("FILES SAVED")
print("=" * 110)
for p in sorted(ARTIFACT_DIR.glob("*")):
    print(p)
print("=" * 110)

[SETUP] Optuna not found. Installing now...


[I 2026-04-03 18:48:09,112] A new study created in memory with name: geo_safety_rf_mlp_bayesian_v1


GEO SAFETY - DYNAMIC BAYESIAN SEARCH (RF + MLP)
[CONFIG] DATA_PATH        : data/compiled_model_ready/MR_cities_worldpop_knn_macro_v5.csv
[CONFIG] TARGET_COL       : safety_index
[CONFIG] TEST_SIZE        : 0.2
[CONFIG] CV_FOLDS         : 5
[CONFIG] N_TRIALS         : 120
[CONFIG] TIMEOUT_SECONDS  : None
[CONFIG] OUTPUT_DIR       : artifacts/geo_safety_optuna_rf_mlp

[STEP 1] DATA LOADED
[INFO] Raw dataframe shape: (509, 77)
[INFO] Total columns      : 77

[STEP 2] AVAILABLE MODEL FEATURES
[INFO] Usable feature count: 45
['dist_nearest_labeled_city', 'log1p_dist_nearest_labeled_city', 'crime_nearest_labeled_city', 'safety_nearest_labeled_city', 'same_country_as_nearest_labeled', 'avg_crime_k5', 'avg_safety_k5', 'avg_crime_k10', 'avg_safety_k10', 'wavg_crime_k5', 'wavg_safety_k5', 'log1p_num_labeled_within_50km', 'log1p_num_labeled_within_100km', 'log1p_num_labeled_within_250km', 'avg_crime_same_country_k5', 'avg_safety_same_country_k5', 'log1p_num_same_country_within_250km', 'log1p_num

[I 2026-04-03 18:48:18,351] Trial 0 finished with value: 12.834197776749619 and parameters: {'model_family': 'mlp', 'feature_set_name': 'geo_only', 'mlp_hidden_layer_sizes': (128, 128), 'mlp_activation': 'tanh', 'mlp_alpha': 0.0002607965659809582, 'mlp_learning_rate_init': 0.0005407232133324001, 'mlp_batch_size': 64, 'mlp_tol': 5.4041038546473305e-05}. Best is trial 0 with value: 12.834197776749619.


[CV] Fold 5/5 | MAE=8.5674 | RMSE=10.6844 | R2=0.4365 | Running mean RMSE=12.8342
[TRIAL 000] COMPLETE
[RESULT] CV mean RMSE: 12.8342
[RESULT] CV std RMSE : 1.6447
[RESULT] CV mean MAE : 10.6093
[RESULT] CV mean R2  : 0.2849

[STUDY UPDATE] Finished trial 0
[STUDY UPDATE] Trial state : TrialState.COMPLETE
[STUDY UPDATE] Trial value : 12.8342
[STUDY UPDATE] Best trial so far : 0
[STUDY UPDATE] Best CV RMSE      : 12.8342
[STUDY UPDATE] Best params       : {'model_family': 'mlp', 'feature_set_name': 'geo_only', 'mlp_hidden_layer_sizes': (128, 128), 'mlp_activation': 'tanh', 'mlp_alpha': 0.0002607965659809582, 'mlp_learning_rate_init': 0.0005407232133324001, 'mlp_batch_size': 64, 'mlp_tol': 5.4041038546473305e-05}

--------------------------------------------------------------------------------------------------------------
[TRIAL 001] START
[INFO] Model family : mlp
[INFO] Feature set  : all_features
[INFO] # Features   : 45
[INFO] Params       : {"hidden_layer_sizes": [128, 64], "activa

...

[CV] Fold 5/5 | MAE=6.2434 | RMSE=8.7930 | R2=0.6183 | Running mean RMSE=8.6892
[TRIAL 117] COMPLETE
[RESULT] CV mean RMSE: 8.6892
[RESULT] CV std RMSE : 0.9452
[RESULT] CV mean MAE : 6.3543
[RESULT] CV mean R2  : 0.6684

[STUDY UPDATE] Finished trial 117
[STUDY UPDATE] Trial state : TrialState.COMPLETE
[STUDY UPDATE] Trial value : 8.6892
[STUDY UPDATE] Best trial so far : 76
[STUDY UPDATE] Best CV RMSE      : 8.6633
[STUDY UPDATE] Best params       : {'model_family': 'rf', 'feature_set_name': 'all_features', 'rf_n_estimators': 300, 'rf_max_depth': 12, 'rf_min_samples_split': 9, 'rf_min_samples_leaf': 2, 'rf_max_features': 0.7, 'rf_bootstrap': True}

--------------------------------------------------------------------------------------------------------------
[TRIAL 118] START
[INFO] Model family : rf
[INFO] Feature set  : all_features
[INFO] # Features   : 45
[INFO] Params       : {"n_estimators": 900, "max_depth": 30, "min_samples_split": 7, "min_samples_leaf": 1, "max_features": 0.9

[I 2026-04-03 18:51:50,288] Trial 118 finished with value: 8.66918767664029 and parameters: {'model_family': 'rf', 'feature_set_name': 'all_features', 'rf_n_estimators': 900, 'rf_max_depth': 30, 'rf_min_samples_split': 7, 'rf_min_samples_leaf': 1, 'rf_max_features': 0.9, 'rf_bootstrap': True}. Best is trial 76 with value: 8.663329717176007.


[CV] Fold 5/5 | MAE=6.2311 | RMSE=8.7710 | R2=0.6203 | Running mean RMSE=8.6692
[TRIAL 118] COMPLETE
[RESULT] CV mean RMSE: 8.6692
[RESULT] CV std RMSE : 0.9655
[RESULT] CV mean MAE : 6.3462
[RESULT] CV mean R2  : 0.6696

[STUDY UPDATE] Finished trial 118
[STUDY UPDATE] Trial state : TrialState.COMPLETE
[STUDY UPDATE] Trial value : 8.6692
[STUDY UPDATE] Best trial so far : 76
[STUDY UPDATE] Best CV RMSE      : 8.6633
[STUDY UPDATE] Best params       : {'model_family': 'rf', 'feature_set_name': 'all_features', 'rf_n_estimators': 300, 'rf_max_depth': 12, 'rf_min_samples_split': 9, 'rf_min_samples_leaf': 2, 'rf_max_features': 0.7, 'rf_bootstrap': True}

--------------------------------------------------------------------------------------------------------------
[TRIAL 119] START
[INFO] Model family : rf
[INFO] Feature set  : all_features
[INFO] # Features   : 45
[INFO] Params       : {"n_estimators": 900, "max_depth": 30, "min_samples_split": 6, "min_samples_leaf": 1, "max_features": 0.9

[I 2026-04-03 18:51:53,814] Trial 119 finished with value: 8.6614104744007 and parameters: {'model_family': 'rf', 'feature_set_name': 'all_features', 'rf_n_estimators': 900, 'rf_max_depth': 30, 'rf_min_samples_split': 6, 'rf_min_samples_leaf': 1, 'rf_max_features': 0.9, 'rf_bootstrap': True}. Best is trial 119 with value: 8.6614104744007.


[CV] Fold 5/5 | MAE=6.2504 | RMSE=8.8044 | R2=0.6174 | Running mean RMSE=8.6614
[TRIAL 119] COMPLETE
[RESULT] CV mean RMSE: 8.6614
[RESULT] CV std RMSE : 0.9531
[RESULT] CV mean MAE : 6.3283
[RESULT] CV mean R2  : 0.6702

[STUDY UPDATE] Finished trial 119
[STUDY UPDATE] Trial state : TrialState.COMPLETE
[STUDY UPDATE] Trial value : 8.6614
[STUDY UPDATE] Best trial so far : 119
[STUDY UPDATE] Best CV RMSE      : 8.6614
[STUDY UPDATE] Best params       : {'model_family': 'rf', 'feature_set_name': 'all_features', 'rf_n_estimators': 900, 'rf_max_depth': 30, 'rf_min_samples_split': 6, 'rf_min_samples_leaf': 1, 'rf_max_features': 0.9, 'rf_bootstrap': True}

[STEP 7] EXPORTING STUDY RESULTS
[INFO] Total completed logged trials: 103
[INFO] Top 15 completed trials by CV RMSE:
 trial_number    state model_family feature_set_name  n_features  uses_current_city_crime_signal  cv_mae_mean  cv_rmse_mean  cv_r2_mean  cv_rmse_std                                                                          